# Clasificación de ObjetosInteractuadosCorrectamente

Este notebook predice la cantidad de objetos interactuados correctamente, con clases enteras de `0` a `9`. El archivo contiene 6000 filas sintéticas trazables a 117 sesiones reales. Se comparan clasificadores tradicionales y una red neuronal con dropout. La evaluación se realiza solo en sesiones reales reservadas, sin sintéticos derivados de ellas.


In [ ]:
# Importaciones y configuración
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
DATA_PATH = Path('neuroflex_dataset_augmented.csv')
print('Configuración lista. Objetivo: ObjetosInteractuadosCorrectamente')


In [ ]:
# Cargar y explorar los datos
df = pd.read_csv(DATA_PATH)
print(f'Dimensiones: {df.shape}')
print('Columnas:', df.columns.tolist())
print('Nulos por columna (solo columnas con nulos):')
print(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False))

if 'age' in df.columns:
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)


In [ ]:
# Definir el objetivo y revisar clases
TARGET = 'ObjetosInteractuadosCorrectamente'
if TARGET not in df.columns:
    raise KeyError(f'No se encontró la columna objetivo {TARGET!r}')

df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce').round().clip(0, 9)
df = df.dropna(subset=[TARGET]).copy()
df['target'] = df[TARGET].astype(int)
class_counts = df.loc[df['source_session_id'].isna(), 'target'].value_counts().reindex(range(10), fill_value=0)
print('Distribución del objetivo en sesiones reales:')
print(class_counts)
ax = class_counts.plot(kind='bar', color='steelblue', edgecolor='black', figsize=(9, 4))
ax.set_title('ObjetosInteractuadosCorrectamente en sesiones reales')
ax.set_xlabel('Cantidad de objetos correctos')
ax.set_ylabel('Frecuencia')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Separar sesiones reales antes de usar filas sintéticas
real_df = df.loc[df['source_session_id'].isna()].copy()
if real_df['session_id'].duplicated().any():
    raise ValueError('session_id debe identificar de forma única cada sesión real')
real_train, real_temp = train_test_split(
    real_df, test_size=0.30, random_state=SEED, stratify=real_df['target']
)
real_val, real_test = train_test_split(
    real_temp, test_size=0.50, random_state=SEED, stratify=real_temp['target']
)
train_source_ids = set(real_train['session_id'])
train_df = df.loc[
    (df['source_session_id'].isna() & df['session_id'].isin(train_source_ids))
    | df['source_session_id'].isin(train_source_ids)
].copy()
print(f'Sesiones reales train: {len(real_train)} | filas train con sintéticos: {len(train_df)}')
print(f'Sesiones reales validación: {len(real_val)} | sesiones reales test: {len(real_test)}')


In [ ]:
# Preparar predictores y mantener evaluación por sesión real
exclude_cols = [
    TARGET, 'target', 'timestamp', 'session_id', 'source_session_id', 'date', 'name',
    'rut', 'patient_updated_at', 'Precision', 'CantAciertasTotales'
]
feature_cols = [c for c in df.columns if c not in exclude_cols]
X_train = train_df[feature_cols].copy(); y_train = train_df['target'].astype(int)
X_val = real_val[feature_cols].copy(); y_val = real_val['target'].astype(int)
X_test = real_test[feature_cols].copy(); y_test = real_test['target'].astype(int)
for col in feature_cols:
    if X_train[col].dtype == 'object' or str(X_train[col].dtype) == 'string':
        converted = pd.to_numeric(X_train[col], errors='coerce')
        if converted.notna().mean() >= 0.8:
            X_train[col] = converted
            X_val[col] = pd.to_numeric(X_val[col], errors='coerce')
            X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
print('Predictores:', feature_cols)
print('Train:', X_train.shape, '| Validación real:', X_val.shape, '| Test real:', X_test.shape)


In [ ]:
# Preprocesamiento numérico y categórico
categorical_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
    ],
    remainder='drop'
)
print(f'Variables numéricas: {len(numeric_cols)} | categóricas: {len(categorical_cols)}')


In [ ]:
# Red neuronal compacta con dropout (no requiere TensorFlow/PyTorch)
class DropoutMLPClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, hidden_layers=(64, 32), dropout_rate=0.25, learning_rate=0.001,
                 epochs=180, batch_size=64, l2=1e-4, random_state=42):
        self.hidden_layers = hidden_layers
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2 = l2
        self.random_state = random_state

    @staticmethod
    def _softmax(logits):
        z = logits - logits.max(axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / exp_z.sum(axis=1, keepdims=True)

    def _forward(self, X, training=False):
        z1 = X @ self.W1 + self.b1
        a1 = np.maximum(z1, 0)
        if training:
            self.mask1 = (self.rng.random(a1.shape) >= self.dropout_rate) / (1 - self.dropout_rate)
            a1 = a1 * self.mask1
        z2 = a1 @ self.W2 + self.b2
        a2 = np.maximum(z2, 0)
        if training:
            self.mask2 = (self.rng.random(a2.shape) >= self.dropout_rate) / (1 - self.dropout_rate)
            a2 = a2 * self.mask2
        logits = a2 @ self.W3 + self.b3
        return logits, a1, a2, z1, z2

    def fit(self, X, y):
        if hasattr(X, 'toarray'):
            X = X.toarray()
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_, encoded = np.unique(y, return_inverse=True)
        n, n_features = X.shape
        self.n_features_in_ = n_features
        n_classes = len(self.classes_)
        if n_classes < 2:
            raise ValueError('Se necesitan al menos dos clases para entrenar.')
        rng = np.random.default_rng(self.random_state)
        self.rng = rng
        h1, h2 = self.hidden_layers
        self.W1 = rng.normal(0, np.sqrt(2/n_features), (n_features, h1))
        self.b1 = np.zeros(h1)
        self.W2 = rng.normal(0, np.sqrt(2/h1), (h1, h2))
        self.b2 = np.zeros(h2)
        self.W3 = rng.normal(0, np.sqrt(2/h2), (h2, n_classes))
        self.b3 = np.zeros(n_classes)
        params = [self.W1, self.b1, self.W2, self.b2, self.W3, self.b3]
        m = [np.zeros_like(v) for v in params]
        v = [np.zeros_like(q) for q in params]
        t = 0
        counts = np.bincount(encoded, minlength=n_classes)
        class_weights = n / (n_classes * counts)
        onehot = np.eye(n_classes)[encoded]

        for _ in range(self.epochs):
            order = rng.permutation(n)
            for start in range(0, n, self.batch_size):
                ids = order[start:start+self.batch_size]
                xb, yb = X[ids], onehot[ids]
                weights = class_weights[encoded[ids]]
                logits, a1, a2, z1, z2 = self._forward(xb, training=True)
                delta3 = (self._softmax(logits) - yb) * weights[:, None] / weights.sum()
                gW3 = a2.T @ delta3 + self.l2 * self.W3
                gb3 = delta3.sum(axis=0)
                da2 = (delta3 @ self.W3.T) * self.mask2
                dz2 = da2 * (z2 > 0)
                gW2 = a1.T @ dz2 + self.l2 * self.W2
                gb2 = dz2.sum(axis=0)
                da1 = (dz2 @ self.W2.T) * self.mask1
                dz1 = da1 * (z1 > 0)
                gW1 = xb.T @ dz1 + self.l2 * self.W1
                gb1 = dz1.sum(axis=0)
                grads = [gW1, gb1, gW2, gb2, gW3, gb3]
                t += 1
                for j, (param, grad) in enumerate(zip(params, grads)):
                    m[j] = 0.9*m[j] + 0.1*grad
                    v[j] = 0.999*v[j] + 0.001*(grad*grad)
                    mhat = m[j] / (1 - 0.9**t)
                    vhat = v[j] / (1 - 0.999**t)
                    param -= self.learning_rate * mhat / (np.sqrt(vhat) + 1e-8)
        return self

    def predict_proba(self, X):
        if hasattr(X, 'toarray'):
            X = X.toarray()
        X = np.asarray(X, dtype=np.float64)
        logits, _, _, _, _ = self._forward(X, training=False)
        return self._softmax(logits)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

# Entrenar y comparar modelos en validación
models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED),
    'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=300, class_weight='balanced', random_state=SEED),
    'GradientBoosting': GradientBoostingClassifier(random_state=SEED),
    'SVC': SVC(class_weight='balanced', random_state=SEED),
    'MLP con Dropout': DropoutMLPClassifier(dropout_rate=0.25, epochs=180, random_state=SEED),
}
results = []
pipelines = {}
for name, estimator in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    row = {
        'model': name,
        'accuracy': accuracy_score(y_val, pred),
        'precision_weighted': precision_score(y_val, pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_val, pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_val, pred, average='weighted', zero_division=0),
    }
    results.append(row)
    pipelines[name] = pipe
    print(f"{name}: accuracy={row['accuracy']:.4f}, F1 ponderado={row['f1_weighted']:.4f}")
results_df = pd.DataFrame(results).sort_values('f1_weighted', ascending=False).reset_index(drop=True)
display(results_df)


In [ ]:
# Comparar métricas de validación
plot_df = results_df.set_index('model')[['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']]
ax = plot_df.plot(kind='bar', figsize=(12, 5), ylim=(0, 1), rot=25)
ax.set_title('Comparación de clasificadores en validación')
ax.set_ylabel('Puntaje')
ax.set_xlabel('Modelo')
plt.legend(['Accuracy', 'Precision ponderado', 'Recall ponderado', 'F1 ponderado'])
plt.tight_layout()
plt.show()


In [ ]:
# Evaluación final del modelo seleccionado en el conjunto de test
best_model_name = results_df.loc[0, 'model']
best_pipe = pipelines[best_model_name]
y_test_pred = best_pipe.predict(X_test)
print('Modelo seleccionado:', best_model_name)
print(f'Accuracy: {accuracy_score(y_test, y_test_pred):.4f}')
print(f'Precision ponderado: {precision_score(y_test, y_test_pred, average="weighted", zero_division=0):.4f}')
print(f'Recall ponderado: {recall_score(y_test, y_test_pred, average="weighted", zero_division=0):.4f}')
print(f'F1 ponderado: {f1_score(y_test, y_test_pred, average="weighted", zero_division=0):.4f}')
print('\nReporte por clase:')
print(classification_report(y_test, y_test_pred, labels=list(range(10)), target_names=[str(i) for i in range(10)], zero_division=0, digits=4))


In [ ]:
# Matriz de confusión
labels = list(range(10))
class_names = [str(i) for i in labels]
cm = confusion_matrix(y_test, y_test_pred, labels=labels)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'Matriz de confusión — {best_model_name} (test)')
plt.xlabel('Cantidad predicha')
plt.ylabel('Cantidad real')
plt.tight_layout()
plt.show()


## Conclusiones

Se predice `ObjetosInteractuadosCorrectamente` en diez clases (`0` a `9`). Se compara una red neuronal de dos capas ocultas con dropout frente a los clasificadores tradicionales. La evaluación usa sesiones reales reservadas; no incluye sintéticos derivados de esas sesiones.

Con diez clases y pocas sesiones reales, los resultados por clase pueden variar bastante. El dropout ayuda a regularizar, pero no garantiza una mejora.
